### Código para preprocesar las partidas (tiempo entre 60 y 300 y dividir en elo alto y medio)

In [1]:
import io
import time
import zstandard as zstd
import re

In [2]:
# --- CONFIGURACIÓN DE RUTAS ---
input_file = "../data/lichess_db_standard_rated_2026-01.pgn.zst"
out_low = "../data/2026_1_preproc_low_elo.pgn.zst"
out_mid = "../data/2026_1_preproc_mid_elo.pgn.zst"
out_high = "../data/2026_1_preproc_high_elo.pgn.zst"
out_higher = "../data/2026_1_preproc_higher_elo.pgn.zst"



In [3]:
total_games = 0
saved_low = 0
saved_mid = 0
saved_high = 0
saved_higher = 0

# Diccionarios para registrar la distribución exacta
conteo_low = {}
conteo_mid = {}
conteo_high = {}
conteo_higher = {}

print(f"Iniciando bifurcación desde: {input_file}")
start_time = time.time()

with open(input_file, "rb") as comp_in, \
     open(out_low, "wb") as comp_out_low, \
     open(out_mid, "wb") as comp_out_mid, \
     open(out_high, "wb") as comp_out_high, \
     open(out_higher, "wb") as comp_out_higher:
     
    dctx = zstd.ZstdDecompressor()
    cctx_low = zstd.ZstdCompressor(level=3)
    cctx_mid = zstd.ZstdCompressor(level=3)
    cctx_high = zstd.ZstdCompressor(level=3)
    cctx_higher = zstd.ZstdCompressor(level=3)
    
    with dctx.stream_reader(comp_in) as reader, \
         cctx_low.stream_writer(comp_out_low) as writer_low, \
         cctx_mid.stream_writer(comp_out_mid) as writer_mid, \
         cctx_high.stream_writer(comp_out_high) as writer_high, \
         cctx_higher.stream_writer(comp_out_higher) as writer_higher:
         
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")
        text_writer_low = io.TextIOWrapper(writer_low, encoding="utf-8")
        text_writer_mid = io.TextIOWrapper(writer_mid, encoding="utf-8")
        text_writer_high = io.TextIOWrapper(writer_high, encoding="utf-8")
        text_writer_higher = io.TextIOWrapper(writer_higher, encoding="utf-8")

        total_games = 0
        buffer = []
        tiempo_val = False
        white_elo = 0
        black_elo = 0
        white_title = ""
        black_title = ""
        
        def procesar_partida():
            global saved_low, saved_mid, saved_high, saved_higher
            # Condición base: que tenga tiempo valido, ambos superen los 1000 y no sean bot

            if buffer and tiempo_val and white_elo >= 1000 and black_elo >= 1000 and white_title != "BOT" and black_title != "BOT":
                game_str = "".join(buffer)
                elo_mean = (white_elo + black_elo)/2
                elo_bin = (elo_mean // 100) * 100
                
                # Bifurcación exclusiva (garantiza que no haya repetidas)
                if elo_mean >= 2500: 
                    text_writer_higher.write(game_str)
                    conteo_higher[elo_bin] = conteo_higher.get(elo_bin, 0) + 1
                    saved_higher += 1
                elif elo_mean >= 2100:
                    text_writer_high.write(game_str)
                    conteo_high[elo_bin] = conteo_high.get(elo_bin, 0) + 1
                    saved_high += 1
                elif elo_mean >= 1300:
                    text_writer_mid.write(game_str)
                    conteo_mid[elo_bin] = conteo_mid.get(elo_bin, 0) + 1
                    saved_mid += 1
                else:
                    text_writer_low.write(game_str)
                    conteo_low[elo_bin] = conteo_low.get(elo_bin, 0) + 1
                    saved_low += 1

        for line in text_stream:
            if line.startswith("[Event "):
                # Evaluar y guardar la partida anterior
                procesar_partida()
                    
                buffer = [line]
                total_games += 1
                tiempo_val = False
                white_elo = 0
                black_elo = 0
                white_title = ""
                black_title = ""
                    
                if total_games > 0 and total_games % 500000 == 0:
                    print(f"  Procesadas {total_games}... Low : {saved_low} | Mid : {saved_mid} | High : {saved_high} | Higher : {saved_higher}")
            else:
                if buffer: 
                    buffer.append(line)
                    if line.startswith("[TimeControl "):
                        match = re.search(r'\[TimeControl "(\d+)', line)
                        if match:
                            tiempo_val = 60 <= int(match.group(1)) <= 300
                    elif line.startswith("[WhiteElo "):
                        try: white_elo = int(line.split('"')[1])
                        except: pass
                    elif line.startswith("[BlackElo "):
                        try: black_elo = int(line.split('"')[1])
                        except: pass
                    elif line.startswith("[WhiteTitle "):
                        try: white_title = line.split('"')[1]
                        except: pass
                    elif line.startswith("[BlackTitle "):
                        try: black_title = line.split('"')[1]
                        except: pass
                        
        # Procesar la última partida del archivo
        procesar_partida()

end_time = time.time()

print("\n--- EXTRACCIÓN Y BIFURCACIÓN FINALIZADAS ---")
print(f"Total de partidas evaluadas: {total_games}")
print(f"Guardadas en {out_low}: {saved_low}")
print(f"Guardadas en {out_mid}: {saved_mid}")
print(f"Guardadas en {out_high}: {saved_high}")
print(f"Guardadas en {out_higher}: {saved_higher}")
print(f"Tiempo de ejecución: {round(end_time - start_time, 2)} segundos")

print("\n--- DISTRIBUCIÓN EN DATASET DE ELO MÁS ELEVADO (2500+) ---")
for elo_bin in sorted(conteo_higher.keys()):
    print(f"Rango {elo_bin}-{elo_bin + 99}: {conteo_higher[elo_bin]} partidas")

print("\n--- DISTRIBUCIÓN EN DATASET DE ELO ELEVADO (2100-2499) ---")
for elo_bin in sorted(conteo_high.keys()):
    print(f"Rango {elo_bin}-{elo_bin + 99}: {conteo_high[elo_bin]} partidas")

print("\n--- DISTRIBUCIÓN EN DATASET MEDIO (1300-2099) ---")
for elo_bin in sorted(conteo_mid.keys()):
    print(f"Rango {elo_bin}-{elo_bin + 99}: {conteo_mid[elo_bin]} partidas")

print("\n--- DISTRIBUCIÓN EN DATASET BAJO (1000-1299) ---")
for elo_bin in sorted(conteo_low.keys()):
    print(f"Rango {elo_bin}-{elo_bin + 99}: {conteo_low[elo_bin]} partidas")

Iniciando bifurcación desde: ../data/lichess_db_standard_rated_2026-01.pgn.zst
  Procesadas 500000... Low : 46120 | Mid : 276602 | High : 55090 | Higher : 7137
  Procesadas 1000000... Low : 90571 | Mid : 549202 | High : 106832 | Higher : 13932
  Procesadas 1500000... Low : 137087 | Mid : 818029 | High : 159595 | Higher : 20930
  Procesadas 2000000... Low : 183947 | Mid : 1087985 | High : 210554 | Higher : 27547
  Procesadas 2500000... Low : 230380 | Mid : 1362535 | High : 266858 | Higher : 34265
  Procesadas 3000000... Low : 280255 | Mid : 1639323 | High : 322350 | Higher : 41624
  Procesadas 3500000... Low : 326155 | Mid : 1920260 | High : 376963 | Higher : 48014
  Procesadas 4000000... Low : 372666 | Mid : 2196883 | High : 428798 | Higher : 54060
  Procesadas 4500000... Low : 419847 | Mid : 2468911 | High : 480167 | Higher : 60485
  Procesadas 5000000... Low : 465898 | Mid : 2741624 | High : 531468 | Higher : 67041
  Procesadas 5500000... Low : 513139 | Mid : 3014326 | High : 582102 